# Diagnóstico del estado inicial de los datos crudos

**CC3084 - Data Science, Proyecto 1: Obtención y Limpieza de datos**

Fuente: Buscador de Establecimientos Educativos, MINEDUC — http://www.mineduc.gob.gt/BUSCAESTABLECIMIENTO_GE/
Filtro aplicado: Nivel Escolar = `DIVERSIFICADO`, todos los departamentos.
Datos obtenidos con `scripts/00_descargar_crudo.py` + `scripts/01_consolidar_crudo.py` (automatizado, ver `docs/01_Diagnostico_y_Plan_de_Limpieza.md`).

Este notebook cubre la **Actividad 3** de la guía: diagnóstico del estado inicial de los datos, respaldado por tablas y estadísticas generadas con código. No se modifica ningún dato aquí — la limpieza (Actividad 5) se hace en un notebook/script separado a partir de lo que este diagnóstico identifica.


In [1]:
import os
import re
from difflib import SequenceMatcher

import pandas as pd

pd.set_option("display.max_rows", 80)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)

RAIZ = ".."
RUTA_CRUDO = os.path.join(RAIZ, "data", "establecimientos_diversificado_crudo.csv")
RUTA_CATALOGO = os.path.join(RAIZ, "data", "catalogo_departamentos_municipios.csv")

df_raw = pd.read_csv(RUTA_CRUDO, dtype=str, keep_default_na=False, encoding="utf-8-sig")
catalogo = pd.read_csv(RUTA_CATALOGO, dtype=str, keep_default_na=False, encoding="utf-8-sig")

COLS_DATO = [c for c in df_raw.columns if c != "ARCHIVO_ORIGEN"]

def es_vacio(valor):
    return valor.strip() == ""

print(f"Archivo crudo: {RUTA_CRUDO}")
print(f"Filas: {df_raw.shape[0]}  |  Columnas: {df_raw.shape[1]}")


Archivo crudo: ..\data\establecimientos_diversificado_crudo.csv
Filas: 11890  |  Columnas: 18


## 0. Filas completamente vacías (artefacto de extracción)

Cada consulta al buscador del MINEDUC devuelve, además de las filas con datos, **una fila extra totalmente vacía al final de la tabla de resultados** (parte de cómo el sitio arma su GridView, no es un error nuestro). Se documenta y se descarta antes de calcular el resto del diagnóstico, porque de lo contrario distorsiona los porcentajes de valores faltantes.


In [2]:
vacias = df_raw[COLS_DATO].apply(lambda col: col.map(es_vacio)).all(axis=1)
print(f"Filas totalmente vacías (una por archivo/departamento consultado): {vacias.sum()}")

df = df_raw.loc[~vacias].copy()
print(f"Filas útiles para el diagnóstico: {len(df)}")


Filas totalmente vacías (una por archivo/departamento consultado): 23
Filas útiles para el diagnóstico: 11867


## a. Número de registros y variables


In [3]:
resumen_dim = pd.DataFrame({
    "concepto": ["Registros (filas)", "Variables originales de la fuente", "Variables totales (incl. ARCHIVO_ORIGEN)"],
    "valor": [len(df), len(COLS_DATO), df.shape[1]],
})
resumen_dim


,concepto,valor
0,Registros (filas),11867
1,Variables originales de la fuente,17
2,Variables totales (incl. ARCHIVO_ORIGEN),18


`ARCHIVO_ORIGEN` no es una variable del fenómeno (no viene del sitio del MINEDUC): es una columna técnica que se agregó durante la descarga para poder rastrear de qué archivo/departamento salió cada fila. Se documentará como tal en el Libro de Códigos y no se cuenta como variable de contenido.


## b. Tipo de dato de cada variable


In [4]:
tipos = pd.DataFrame({
    "variable": COLS_DATO,
    "dtype_actual": [df[c].dtype for c in COLS_DATO],
    "ejemplo": [df[c].iloc[df[c].str.strip().ne("").idxmax()] if df[c].str.strip().ne("").any() else "" for c in COLS_DATO],
})
tipos


,variable,dtype_actual,ejemplo
0,CODIGO,object,00-01-0158-46
1,DISTRITO,object,01-01-0026
2,DEPARTAMENTO,object,CIUDAD CAPITAL
3,MUNICIPIO,object,ZONA 1
4,ESTABLECIMIENTO,object,COLEGIO TECNICO PROGRESISTA CETECPRO
5,DIRECCION,object,2A. AVENIDA 3-59
6,TELEFONO,object,22512759
7,SUPERVISOR,object,LUDIBEL AMARILIS GALICIA DE FLORES
8,DIRECTOR,object,IRMA AMPARO TOLEDO HERNANDEZ
9,NIVEL,object,DIVERSIFICADO


Todas las variables llegan como texto (`object`), incluyendo `CODIGO` y `DISTRITO`, que son identificadores estructurados (no cantidades) y **deben permanecer como texto** aun después de la limpieza, para no perder ceros a la izquierda (ej. `03-01-0052-46`). No hay variables verdaderamente numéricas en este dataset ni fechas en esta etapa. El problema real de tipo no es "número guardado como texto" sino **variables categóricas que deberían tener un dominio cerrado y controlado** (`SECTOR`, `AREA`, `STATUS`, `MODALIDAD`, `JORNADA`, `PLAN`, `NIVEL`, `DEPARTAMENTO`) — se verifican en la sección f.


## c. Cantidad y porcentaje de valores faltantes por variable

Se cuenta como faltante: celda vacía o que solo contiene espacios en blanco (no se imputa nada; los placeholders tipo `"--"` se identifican aparte, en la sección f/h, porque técnicamente no son cadenas vacías pero funcionan como faltante disfrazado).


In [5]:
filas_na = []
for c in COLS_DATO:
    faltantes = df[c].map(es_vacio).sum()
    filas_na.append((c, faltantes, round(100 * faltantes / len(df), 2)))

tabla_na = pd.DataFrame(filas_na, columns=["variable", "faltantes", "pct_faltante"]).sort_values("pct_faltante", ascending=False)
tabla_na


,variable,faltantes,pct_faltante
8,DIRECTOR,1732,14.60
6,TELEFONO,946,7.97
7,SUPERVISOR,535,4.51
1,DISTRITO,532,4.48
5,DIRECCION,76,0.64
4,ESTABLECIMIENTO,5,0.04
0,CODIGO,0,0.00
3,MUNICIPIO,0,0.00
2,DEPARTAMENTO,0,0.00
9,NIVEL,0,0.00


In [6]:
total_celdas = len(df) * len(COLS_DATO)
total_faltantes = tabla_na["faltantes"].sum()
variables_con_na = (tabla_na["faltantes"] > 0).sum()
print(f"Celdas totales: {total_celdas}")
print(f"Celdas faltantes: {total_faltantes} ({100*total_faltantes/total_celdas:.2f}% del total de celdas)")
print(f"Variables con al menos un faltante: {variables_con_na} de {len(COLS_DATO)}")


Celdas totales: 201739
Celdas faltantes: 3826 (1.90% del total de celdas)
Variables con al menos un faltante: 6 de 17


## d. Cantidad de valores únicos por variable


In [7]:
tabla_unicos = pd.DataFrame({
    "variable": COLS_DATO,
    "valores_unicos": [df[c].nunique() for c in COLS_DATO],
    "pct_del_total": [round(100 * df[c].nunique() / len(df), 2) for c in COLS_DATO],
}).sort_values("valores_unicos", ascending=False)
tabla_unicos


,variable,valores_unicos,pct_del_total
0,CODIGO,11867,100.00
5,DIRECCION,7527,63.43
4,ESTABLECIMIENTO,6668,56.19
6,TELEFONO,6573,55.39
8,DIRECTOR,5563,46.88
1,DISTRITO,1682,14.17
7,SUPERVISOR,1286,10.84
3,MUNICIPIO,352,2.97
16,DEPARTAMENTAL,26,0.22
2,DEPARTAMENTO,23,0.19


`CODIGO` tiene tantos valores únicos como filas (es la llave del dataset). Las variables con muy pocos valores únicos (`NIVEL`, `MODALIDAD`, `SECTOR`, `AREA`, `STATUS`) son candidatas naturales a dominio cerrado y se validan en la sección f. `MUNICIPIO` tiene 352 valores únicos; se valida contra el catálogo oficial del propio sitio en la sección f.2 (el catálogo completo tiene 363 municipios en sus 23 "departamentos", pero no todos tienen necesariamente un establecimiento de nivel Diversificado registrado, de ahí que 352 sea menor).


## e. Registros duplicados exactos


In [8]:
dup_exactos_mask = df.duplicated(subset=COLS_DATO, keep=False)
dup_codigo_mask = df["CODIGO"].duplicated(keep=False) & (df["CODIGO"].str.strip() != "")

resumen_dup = pd.DataFrame({
    "tipo": ["Duplicados exactos (todas las columnas iguales)", "CODIGO repetido (debería ser llave única)"],
    "filas_involucradas": [dup_exactos_mask.sum(), dup_codigo_mask.sum()],
})
resumen_dup


,tipo,filas_involucradas
0,Duplicados exactos (todas las columnas iguales),0
1,CODIGO repetido (debería ser llave única),0


No hay duplicados exactos ni `CODIGO` repetidos en el crudo consolidado. Esto **no significa que no haya duplicados** — como se documenta en la sección h, el problema de duplicidad en este dataset es de **duplicados parciales** (mismo colegio escrito de forma distinta, o con jornadas/planes legítimamente distintos que no deben fusionarse), que se detectan con similitud de cadenas, no con igualdad exacta.


## f. Variables con valores fuera de dominio o inconsistentes

### f.1 Variables de dominio cerrado (deberían tener un catálogo fijo de valores)


In [9]:
for col in ["NIVEL", "SECTOR", "AREA", "STATUS", "MODALIDAD", "JORNADA", "PLAN"]:
    vals = sorted(v for v in df[col].unique() if v.strip() != "")
    print(f"-- {col} ({len(vals)} valores distintos) --")
    print(vals)
    print()


-- NIVEL (1 valores distintos) --
['DIVERSIFICADO']

-- SECTOR (4 valores distintos) --
['COOPERATIVA', 'MUNICIPAL', 'OFICIAL', 'PRIVADO']

-- AREA (3 valores distintos) --
['RURAL', 'SIN ESPECIFICAR', 'URBANA']

-- STATUS (5 valores distintos) --
['ABIERTA', 'CERRADA DEFINITIVAMENTE', 'CERRADA TEMPORALMENTE', 'TEMPORAL NOMBRAMIENTO', 'TEMPORAL TITULOS']

-- MODALIDAD (2 valores distintos) --
['BILINGUE', 'MONOLINGUE']

-- JORNADA (6 valores distintos) --
['DOBLE', 'INTERMEDIA', 'MATUTINA', 'NOCTURNA', 'SIN JORNADA', 'VESPERTINA']

-- PLAN (13 valores distintos) --
['A DISTANCIA', 'DIARIO(REGULAR)', 'DOMINICAL', 'FIN DE SEMANA', 'INTERCALADO', 'IRREGULAR', 'MIXTO', 'SABATINO', 'SEMIPRESENCIAL', 'SEMIPRESENCIAL (DOS DÍAS A LA SEMANA)', 'SEMIPRESENCIAL (FIN DE SEMANA)', 'SEMIPRESENCIAL (UN DÍA A LA SEMANA)', 'VIRTUAL A DISTANCIA']



`NIVEL`, `SECTOR`, `AREA`, `STATUS`, `MODALIDAD` y `JORNADA` tienen un número pequeño y estable de categorías, todas en mayúsculas y sin variantes de escritura evidentes — no se detectan valores fuera de dominio en ellas. `PLAN` es la excepción: 13 categorías donde varias formas de `SEMIPRESENCIAL` conviven (`SEMIPRESENCIAL`, `SEMIPRESENCIAL (UN DÍA A LA SEMANA)`, `SEMIPRESENCIAL (FIN DE SEMANA)`, `SEMIPRESENCIAL (DOS DÍAS A LA SEMANA)`) — no está claro todavía si son redundantes o representan modalidades genuinamente distintas; se marca como pendiente de decisión en el plan de limpieza, no se unifica aquí.


### f.2 DEPARTAMENTO y MUNICIPIO contra el catálogo oficial del propio MINEDUC

El catálogo (`data/catalogo_departamentos_municipios.csv`) se descargó directamente del combo Departamento → Municipio del mismo sitio (`scripts/00b_descargar_catalogo.py`), para no depender de una lista tipeada a mano.


In [10]:
deptos_catalogo = set(catalogo["DEPARTAMENTO"].str.upper().str.strip())
deptos_datos = set(df["DEPARTAMENTO"].str.upper().str.strip()) - {""}
print("Departamentos en los datos que NO están en el catálogo:", deptos_datos - deptos_catalogo)
print("Departamentos del catálogo nunca vistos en los datos:", deptos_catalogo - deptos_datos)


Departamentos en los datos que NO están en el catálogo: set()
Departamentos del catálogo nunca vistos en los datos: set()


In [11]:
pares_catalogo = set(
    zip(catalogo["DEPARTAMENTO"].str.upper().str.strip(), catalogo["MUNICIPIO"].str.upper().str.strip())
)
pares_datos = df.loc[
    df["DEPARTAMENTO"].str.strip().ne("") & df["MUNICIPIO"].str.strip().ne(""),
    ["DEPARTAMENTO", "MUNICIPIO"],
].apply(lambda r: (r["DEPARTAMENTO"].upper().strip(), r["MUNICIPIO"].upper().strip()), axis=1)

fuera_de_catalogo = pares_datos[~pares_datos.isin(pares_catalogo)]
print(f"Filas cuyo par (DEPARTAMENTO, MUNICIPIO) no aparece en el catálogo: {len(fuera_de_catalogo)}")
if len(fuera_de_catalogo):
    tabla_fuera_catalogo = pd.DataFrame(fuera_de_catalogo.tolist(), columns=["departamento", "municipio"])
    tabla_fuera_catalogo = (
        tabla_fuera_catalogo.value_counts().reset_index(name="filas").sort_values("filas", ascending=False)
    )
else:
    tabla_fuera_catalogo = pd.DataFrame(columns=["departamento", "municipio", "filas"])
tabla_fuera_catalogo


Filas cuyo par (DEPARTAMENTO, MUNICIPIO) no aparece en el catálogo: 0


,departamento,municipio,filas


**Todas** las filas tienen un par (DEPARTAMENTO, MUNICIPIO) que existe en el catálogo del propio MINEDUC: no hay municipios inventados, mal escritos ni fuera del departamento que les corresponde. Esto es de esperarse porque `DEPARTAMENTO` y `MUNICIPIO` se llenan en el sitio original a partir de los mismos combos `<select>` (no son texto libre digitado por un usuario) — a diferencia de `ESTABLECIMIENTO`, `DIRECCION`, `SUPERVISOR` o `DIRECTOR`, que sí son campos de texto libre y por eso concentran los problemas de formato reales (secciones g y h).


### f.3 Formato y unicidad de CODIGO


In [12]:
patron_codigo = re.compile(r"^\d{2}-\d{2}-\d{4}-\d{2}$")
codigo_ok = df["CODIGO"].map(lambda x: bool(patron_codigo.match(x.strip())))
print(f"CODIGO que cumple el patrón ##-##-####-##: {codigo_ok.sum()} / {len(df)}")
print("Ejemplos que NO cumplen:")
print(df.loc[~codigo_ok, "CODIGO"].drop_duplicates().to_string(index=False) or "(ninguno)")


CODIGO que cumple el patrón ##-##-####-##: 11867 / 11867
Ejemplos que NO cumplen:
Series([], )


### f.4 Consistencia entre CODIGO y DEPARTAMENTO (dígitos iniciales del código)


In [13]:
depto_por_prefijo = {
    "01": "GUATEMALA", "00": "CIUDAD CAPITAL", "02": "EL PROGRESO", "03": "SACATEPEQUEZ",
    "04": "CHIMALTENANGO", "05": "ESCUINTLA", "06": "SANTA ROSA", "07": "SOLOLA",
    "08": "TOTONICAPAN", "09": "QUETZALTENANGO", "10": "SUCHITEPEQUEZ", "11": "RETALHULEU",
    "12": "SAN MARCOS", "13": "HUEHUETENANGO", "14": "QUICHE", "15": "BAJA VERAPAZ",
    "16": "ALTA VERAPAZ", "17": "PETEN", "18": "IZABAL", "19": "ZACAPA", "20": "CHIQUIMULA",
    "21": "JALAPA", "22": "JUTIAPA",
}
prefijo = df["CODIGO"].str.slice(0, 2)
esperado = prefijo.map(depto_por_prefijo)
inconsistente = esperado.notna() & (esperado != df["DEPARTAMENTO"].str.strip().str.upper())
print(f"Filas donde el prefijo de CODIGO no coincide con DEPARTAMENTO: {inconsistente.sum()}")
df.loc[inconsistente, ["CODIGO", "DEPARTAMENTO"]].drop_duplicates(subset=["DEPARTAMENTO"])


Filas donde el prefijo de CODIGO no coincide con DEPARTAMENTO: 0


,CODIGO,DEPARTAMENTO


El mapa de prefijos usado arriba **respeta la convención administrativa** documentada en `docs/01_Diagnostico_y_Plan_de_Limpieza.md`: el MINEDUC trata `CIUDAD CAPITAL` (prefijo `00`) como una entidad separada de `GUATEMALA` (prefijo `01`), aunque ambas correspondan al mismo departamento oficial del país. Tratándolas como categorías distintas (que es como realmente aparecen en el campo `DEPARTAMENTO`), el prefijo de `CODIGO` es **100% consistente** con `DEPARTAMENTO` — no hay ninguna fila real mal etiquetada. (Si en cambio se colapsara `00` y `01` en una sola "GUATEMALA", como hace un cruce ingenuo, aparecerían ~2,161 falsos positivos: son exactamente las filas de Ciudad Capital, no errores de digitación.)


In [14]:
otros_inconsistentes = df.loc[inconsistente]
print(f"Filas con prefijo de CODIGO inconsistente con DEPARTAMENTO: {len(otros_inconsistentes)}")


Filas con prefijo de CODIGO inconsistente con DEPARTAMENTO: 0


### f.5 TELEFONO fuera de dominio (no numérico, longitud incorrecta, múltiples valores en una celda)


In [15]:
tel = df["TELEFONO"].str.strip()
con_letras = tel.str.contains(r"[A-Za-z]", regex=True, na=False)
multiples = tel.str.contains(r"[-/,]", regex=True, na=False)
solo_digitos_8 = tel.str.fullmatch(r"\d{8}")
cortos = tel.str.replace(r"\D", "", regex=True).str.len().lt(8) & (tel != "")

tabla_tel = pd.DataFrame({
    "condicion": [
        "Formato estándar (8 dígitos, sin más)",
        "Contiene letras",
        "Más de un teléfono en la celda (separado por -, / o ,)",
        "Menos de 8 dígitos (sin contar vacíos)",
        "Vacío",
    ],
    "filas": [
        solo_digitos_8.sum(), con_letras.sum(), multiples.sum(), cortos.sum(), tel.eq("").sum(),
    ],
})
tabla_tel


,condicion,filas
0,"Formato estándar (8 dígitos, sin más)",10670
1,Contiene letras,9
2,"Más de un teléfono en la celda (separado por -, / o ,)",184
3,Menos de 8 dígitos (sin contar vacíos),49
4,Vacío,946


## g. Variables con formatos inconsistentes

### g.1 DISTRITO: conviven varios formatos en el mismo campo


In [16]:
distrito = df["DISTRITO"].str.strip()

def clasificar_distrito(v):
    if v == "":
        return "vacío"
    if re.fullmatch(r"\d{2}-\d{3}", v):
        return "DD-DDD"
    if re.fullmatch(r"\d{2}-\d{2}-\d{4}", v):
        return "DD-DD-DDDD"
    return "otro formato"

tabla_distrito = distrito.map(clasificar_distrito).value_counts().rename_axis("formato").reset_index(name="filas")
tabla_distrito


,formato,filas
0,DD-DD-DDDD,6226
1,DD-DDD,5039
2,vacío,532
3,otro formato,70


### g.2 Normalización de texto: espacios y placeholders en campos libres


In [17]:
filas_norm = []
for col in ["ESTABLECIMIENTO", "DIRECCION", "MUNICIPIO", "DEPARTAMENTO", "SUPERVISOR", "DIRECTOR"]:
    s = df[col]
    espacio_extremo = (s.str.strip() != s).sum()
    espacio_doble = s.str.contains(r"  ", regex=True, na=False).sum()
    placeholder = (s.str.strip() == "--").sum()
    comillas = s.str.contains(r"['\u2018\u2019\"]", regex=True, na=False).sum()
    filas_norm.append((col, espacio_extremo, espacio_doble, placeholder, comillas))

tabla_norm = pd.DataFrame(
    filas_norm,
    columns=["variable", "espacios_inicio_fin", "espacios_dobles", "placeholder_--", "contiene_comillas_o_apostrofe"],
)
tabla_norm


,variable,espacios_inicio_fin,espacios_dobles,placeholder_--,contiene_comillas_o_apostrofe
0,ESTABLECIMIENTO,0,1392,0,2959
1,DIRECCION,0,485,1,388
2,MUNICIPIO,0,0,0,0
3,DEPARTAMENTO,0,0,0,0
4,SUPERVISOR,0,98,0,0
5,DIRECTOR,0,866,41,1


No hay espacios al inicio/final en ningún campo del crudo consolidado — pero esto es en parte un efecto del método de extracción (el parser de la tabla ya colapsa espacios al leer cada celda), no necesariamente evidencia de que la fuente original estuviera así de limpia; se documenta como limitación metodológica en `docs/01_Diagnostico_y_Plan_de_Limpieza.md`. Los espacios dobles internos, en cambio, sí sobrevivieron y son reales.


### g.3 PLAN: categorías que probablemente representan lo mismo escrito de forma distinta


In [18]:
plan_counts = df["PLAN"].value_counts()
plan_counts


PLAN
DIARIO(REGULAR)                          7452
FIN DE SEMANA                            2898
SEMIPRESENCIAL (FIN DE SEMANA)            542
SEMIPRESENCIAL (UN DÍA A LA SEMANA)       437
A DISTANCIA                               188
SEMIPRESENCIAL                            118
VIRTUAL A DISTANCIA                        70
SEMIPRESENCIAL (DOS DÍAS A LA SEMANA)      67
SABATINO                                   59
DOMINICAL                                  27
MIXTO                                       4
IRREGULAR                                   3
INTERCALADO                                 2
Name: count, dtype: int64

## h. Identificación de problemas potenciales de calidad de datos

Síntesis de todo lo anterior, más dos hallazgos adicionales que no encajan en las secciones a-g: **posibles duplicados parciales** (mismo colegio, nombre escrito distinto) y la mezcla de **comillas decorativas vs. apóstrofes ortográficos legítimos** en nombres de origen maya dentro de `ESTABLECIMIENTO`.


### h.1 Muestra de posibles duplicados parciales (similitud de nombre dentro del mismo municipio)

Se compara `ESTABLECIMIENTO` por pares dentro de cada `MUNICIPIO` (mismo municipio, para mantener la comparación acotada y relevante) usando `difflib.SequenceMatcher` como métrica de similitud rápida. Esto es solo un **conteo exploratorio** para dimensionar el problema en el diagnóstico — la técnica formal (Jaro-Winkler/RapidFuzz) y la revisión caso por caso se hacen en la Actividad 5 (limpieza), como exige la guía; aquí no se fusiona ni se descarta ninguna fila.


In [19]:
def similares(nombres, umbral=0.92):
    candidatos = []
    n = len(nombres)
    for i in range(n):
        for j in range(i + 1, n):
            a, b = nombres[i], nombres[j]
            if a == b:
                continue
            ratio = SequenceMatcher(None, a, b).ratio()
            if ratio >= umbral:
                candidatos.append((a, b, round(ratio, 3)))
    return candidatos

pares_similares = []
for municipio, grupo in df.groupby("MUNICIPIO"):
    nombres = grupo["ESTABLECIMIENTO"].str.strip().unique().tolist()
    if len(nombres) < 2 or len(nombres) > 400:
        continue
    for a, b, ratio in similares(nombres):
        pares_similares.append((municipio, a, b, ratio))

tabla_similares = pd.DataFrame(pares_similares, columns=["municipio", "establecimiento_1", "establecimiento_2", "similitud"])
print(f"Pares de nombres de establecimiento muy similares (posible duplicado parcial), dentro del mismo municipio: {len(tabla_similares)}")
tabla_similares.sort_values("similitud", ascending=False).head(20)


Pares de nombres de establecimiento muy similares (posible duplicado parcial), dentro del mismo municipio: 1814


,municipio,establecimiento_1,establecimiento_2,similitud
574,LIVINGSTON,COLEGIO DE EDUCACION BASICA Y DIVERSIFICADA EN ESTUDIOS COMERCIALES CON ORIE...,COLEGIO DE EDUCACION BASICA Y DIVERSIFICADA EN ESTUDIOS COMERCIALES CON ORIE...,0.996
296,ESCUINTLA,"INSTITUTO MIXTO PARTICULAR DE EDUCACIÓN BÁSICA Y DIVERSIFICADA ""ALMA LIBERTA...","INSTITUTO MIXTO PARTICULAR DE EDUCACIÓN BÁSICA Y DIVERSIFICADA ""ALMA LIBERTA...",0.995
1017,SAN FRANCISCO,"INSTITUTO NACIONAL DIVERSIFICADO DE BACHILLERATO INDUSTRIAL ""LIC. RODERICO S...","INSTITUTO NACIONAL DIVERSIFICADO DE BACHILLERATO INDUSTRIAL ""LIC. RODERICO ...",0.995
110,CHIMALTENANGO,CENTRO DE ESTUDIOS TÉCNICOS Y AVANZADOS DE CHIMALTENANGO NO. 2 (C.E.T.A.CH.N...,CENTRO DE ESTUDIOS TÉCNICOS Y AVANZADOS DE CHIMALTENANGO NO. 2 (C.E.T.A.CH. ...,0.994
805,PUERTO BARRIOS,COLEGIO EDUCATIVO PRIVADO MIXTO DE FORMACION INTEGRAL EN COMPUTACION CEIF COMP,"COLEGIO EDUCATIVO PRIVADO MIXTO DE FORMACION INTEGRAL EN COMPUTACION, CEIF COMP",0.994
537,LA LIBERTAD,"COLEGIO DE EDUCACION INTEGRAL CON ORIENTACION UNIVERSITARIA ""EL PODER DEL SA...","COLEGIO DE EDUCACION INTEGRAL CON ORIENTACION UNIVERSITARIA, ""EL PODER DEL S...",0.994
932,SAN AGUSTIN ACASAGUASTLAN,"COLEGIO ""CENTRO DE FORMACION PROFESIONAL PRE-UNIVERSITARIA DEL CICLO DIVERSI...","COLEGIO ""CENTRO DE FORMACION PROFESIONAL PREUNIVERSITARIA DEL CICLO DIVERSIF...",0.994
106,CHIMALTENANGO,"CENTRO DE ESTUDIOS TECNICOS Y AVANZADOS DE CHIMALTENANGO ""C.E.T.A.CH.""","CENTRO DE ESTUDIOS TECNICOS Y AVANZADOS DE CHIMALTENANGO ""C.E.T.A.CH. """,0.993
292,ESCUINTLA,INSTITUTO PARTICULAR MIXTO DE EDUCACIÓN BÁSICA Y BACHILLERATO POR MADUREZ,INSTITUTO PARTICULAR MIXTO DE EDUCACIÓN BÁSICA Y BACHILLERATO POR MADUREZ,0.993
285,ESCUINTLA,INSTITUTO PARTICULAR MIXTO DE EDUCACIÓN BÁSICA Y BACHILLERATO POR MADUREZ,INSTITUTO PARTICULAR MIXTO DE EDUCACIÓN BÁSICA Y BACHILLERATO POR MADUREZ,0.993


### h.2 Comillas/apóstrofes en ESTABLECIMIENTO: hay que distinguir dos fenómenos distintos


In [20]:
con_comilla_doble = df["ESTABLECIMIENTO"].str.contains('"', regex=False, na=False)
con_apostrofe = df["ESTABLECIMIENTO"].str.contains("'", regex=False, na=False)
print(f"Contienen comillas dobles (probable alias/apodo entre comillas): {con_comilla_doble.sum()}")
print(f"Contienen apóstrofe simple: {con_apostrofe.sum()}")
print()
print("Ejemplos con apóstrofe (mezcla nombres de origen k'iche' legítimos con posibles comillas decorativas):")
print(df.loc[con_apostrofe, "ESTABLECIMIENTO"].drop_duplicates().head(10).to_string(index=False))


Contienen comillas dobles (probable alias/apodo entre comillas): 2225
Contienen apóstrofe simple: 747

Ejemplos con apóstrofe (mezcla nombres de origen k'iche' legítimos con posibles comillas decorativas):
INSTITUTO DE EDUCACION DIVERSIFICADA 'CENTRO DE ESTUDIOS MERCADOLOGICOS Y PUB...
                     INSTITUTO DE EDUCACION DIVERSIFICADA 'LICEO MARIANO GALVEZ'
                                                 INSTITUTO NORMAL 'CASA CENTRAL'
INSTITUTO PRIVADO PARA SEÑORITAS DE EDUCACION DIVERSIFICADA 'SPEEDWRITING' (E...
INSTITUTO PRIVADO PARA SEÑORITAS DE EDUCACION DIVERSIFICADA 'ACADEMIA PRACTIC...
                                                      INSTITUTO 'FEDERICO CROWE'
   INSTITUTO PRIVADO MIXTO DE EDUCACION DIVERSIFICADA 'BROW'S CENTRO DE LENGUAS'
INSTITUTO PRIVADO MIXTO DE EDUCACION DIVERSIFICADA 'TECNICO CENTROAMERICANO D...
         INSTITUTO DE EDUCACION DIVERSIFICADA COLEGIO 'SAN JOSE DE LOS INFANTES'
INSTITUTO PRIVADO PARA SEÑORITAS DE EDUCACION DIVERSIFICADA 'BILI

### h.3 Placeholder `"--"` como faltante disfrazado (no cuenta como celda vacía en la sección c)


In [21]:
placeholders = {col: (df[col].str.strip() == "--").sum() for col in COLS_DATO}
tabla_placeholder = pd.DataFrame(
    [(k, v) for k, v in placeholders.items() if v > 0], columns=["variable", "filas_con_placeholder_--"]
)
tabla_placeholder


,variable,filas_con_placeholder_--
0,DIRECCION,1
1,DIRECTOR,41


### h.4 Resumen ejecutivo del diagnóstico


In [22]:
resumen_hallazgos = pd.DataFrame([
    ("Filas 100% vacías (artefacto de extracción, ya descartadas del resto del diagnóstico)", vacias.sum()),
    ("Duplicados exactos", int(dup_exactos_mask.sum())),
    ("CODIGO repetido", int(dup_codigo_mask.sum())),
    ("Posibles duplicados parciales (nombre similar, mismo municipio)", len(tabla_similares)),
    ("Filas (DEPARTAMENTO, MUNICIPIO) fuera del catálogo oficial del sitio", len(fuera_de_catalogo)),
    ("CODIGO que no cumple el patrón ##-##-####-##", int((~codigo_ok).sum())),
    ("Filas con prefijo de CODIGO inconsistente con DEPARTAMENTO", len(otros_inconsistentes)),
    ("TELEFONO con letras", int(con_letras.sum())),
    ("TELEFONO con más de un número en la celda", int(multiples.sum())),
    ("TELEFONO con menos de 8 dígitos", int(cortos.sum())),
    ("DIRECTOR/otros campos con placeholder \"--\"", int(tabla_placeholder["filas_con_placeholder_--"].sum()) if not tabla_placeholder.empty else 0),
    ("Variables con al menos un valor faltante", int(variables_con_na)),
    ("Categorías de PLAN potencialmente redundantes (variantes de SEMIPRESENCIAL)", 4),
    ("Formatos distintos conviviendo en DISTRITO", int((tabla_distrito['formato'] != 'vacío').sum())),
], columns=["hallazgo", "cantidad"])
resumen_hallazgos


,hallazgo,cantidad
0,"Filas 100% vacías (artefacto de extracción, ya descartadas del resto del dia...",23
1,Duplicados exactos,0
2,CODIGO repetido,0
3,"Posibles duplicados parciales (nombre similar, mismo municipio)",1814
4,"Filas (DEPARTAMENTO, MUNICIPIO) fuera del catálogo oficial del sitio",0
5,CODIGO que no cumple el patrón ##-##-####-##,0
6,Filas con prefijo de CODIGO inconsistente con DEPARTAMENTO,0
7,TELEFONO con letras,9
8,TELEFONO con más de un número en la celda,184
9,TELEFONO con menos de 8 dígitos,49


Este resumen es el insumo directo para la **Actividad 4 (Plan de limpieza)**, que ya se desarrolló variable por variable en `docs/01_Diagnostico_y_Plan_de_Limpieza.md`. Ningún dato se modificó en este notebook — es puramente diagnóstico.
